# 01. Khám Phá Dữ Liệu Biển Báo Giao Thông (Dataset Exploration)

Notebook này khám phá và phân tích tập dữ liệu biển báo giao thông Việt Nam:
1. Cấu trúc thư mục dữ liệu và nhãn nhị phân / đa lớp.
2. Phân bố các lớp biển báo (Class Distribution) và hiện tượng mất cân bằng lớp.
3. Phân bố kích thước ảnh và tỷ lệ khung hình của biển báo.
4. Phân tích màu sắc trên không gian màu HSV (Đỏ, Vàng, Xanh dương).

In [ ]:
from pathlib import Path
from collections import Counter
import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_config, list_image_paths, load_image
from src.utils import read_label_boxes
from src.segmentation import segment_colors

## 1. Nạp Cấu Hình và Đọc Danh Sách Nhãn Lớp

In [ ]:
config, project_root, _ = load_config()
data_dir = project_root / config["paths"]["data_raw"]
images_dir = data_dir / "images"
labels_dir = data_dir / "labels"

classes_file = data_dir / "classes_vie.txt"
class_names = [line.strip() for line in classes_file.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"Tổng số lớp biển báo khai báo: {len(class_names)}")
image_paths = list_image_paths(images_dir)
print(f"Số lượng ảnh mẫu: {len(image_paths)}")

## 2. Thống Kê Phân Bố Kích Thước Bounding Box

In [ ]:
widths, heights, aspect_ratios, class_counts = [], [], [], Counter()

for img_p in image_paths:
    img = load_image(img_p)
    if img is None:
        continue
    lbl_p = labels_dir / f"{img_p.stem}.txt"
    if not lbl_p.is_file():
        continue
    boxes = read_label_boxes(lbl_p, img.shape)
    for b in boxes:
        w, h = b["w"], b["h"]
        widths.append(w)
        heights.append(h)
        if h > 0:
            aspect_ratios.append(w / float(h))
        if b["class"] is not None:
            class_counts[b["class"]] += 1

print(f"Tổng số bounding box tìm thấy: {len(widths)}")
if widths:
    print(f"Kích thước trung bình: {np.mean(widths):.1f}x{np.mean(heights):.1f} px")
    print(f"Tỷ lệ khung hình trung bình (w/h): {np.mean(aspect_ratios):.2f}")

## 3. Trực Quan Hóa Phân Đoạn Màu Sắc HSV

In [ ]:
if image_paths:
    sample_img = load_image(image_paths[0])
    segmented, mask = segment_colors(sample_img)
    print(f"Mặt nạ màu HSV kích thước: {mask.shape}, Tỷ lệ điểm màu: {np.mean(mask > 0)*100:.2f}%")